In [63]:
from dataclasses import dataclass
from typing import Optional
from typing import Literal
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel
import os
import json

load_dotenv()

print("GROQ:", bool(os.getenv("GROQ_API_KEY")))
print("OPENROUTER:", bool(os.getenv("OPENROUTER_API_KEY")))

@dataclass(frozen=True)
class Provider:
    """ One Provider to reliably route request across all inference providers"""
    name: str
    env_var: str
    is_free: bool
    base_url: Optional[str]
    model: Optional[str]

PROVIDERS = [
    Provider("Groq", "GROQ_API_KEY", True, "https://api.groq.com/openai/v1", "openai/gpt-oss-120b"),
    Provider("OpenRouter", "OPENROUTER_API_KEY", True, "https://openrouter.ai/api/v1", "openrouter/free"),
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if(os.getenv(provider.env_var)):
            return provider;

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider API Key is set add one of the {expected} to your environment variables")


def build_client(provider: Provider) -> OpenAI:
    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )

def have_any_key() -> bool:
    return any(os.getenv(provider.env_var) for provider in PROVIDERS)


print("Found a provider key" if have_any_key() else "No Provider key found")

GROQ: True
OPENROUTER: True
Found a provider key


In [64]:
SYSTEM_PROMPT = """
You are a product review sentiment analysis expert.

You are given a product review and you need to analyze the sentiment of the review.

I need you to analyze the review and return the sentiment of the review in a JSON format.

Do not provide hidden chain of thought or detailed internal reasoning.

The JSON should have the exact following fields:
- sentiment: The sentiment of the review.
- confidence: The confidence of the sentiment.
- reasoning: The reasoning for the sentiment in one sentence.

""".strip()

In [65]:
def analyze_review(review: str) -> str:
    """
    Analyze the sentiment of the review.
    """
    provider = select_provider()
    client = build_client(provider)

    response = client.chat.completions.create(
        model = provider.model,
        max_completion_tokens=300,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": review
            }
        ],
    )

    return response.choices[0].message.content

In [59]:
REVIEW = [
    "I love this product! It's amazing and I'm so happy with it.",
    "I'm so disappointed with this product. It's not worth the money.",
    "This product is okay. I don't really like it.",
    "I'm not sure about this product. It's good but I'm not sure if I'll buy it again.",
]


In [66]:
review1 = analyze_review(REVIEW[0])
print(review1)
review2 = analyze_review(REVIEW[1])
print(review2)
review3 = analyze_review(REVIEW[2])
print(review3)
review4 = analyze_review(REVIEW[3])

{
  "sentiment": "positive",
  "confidence": 0.99,
  "reasoning": "The review uses enthusiastic language such as \"love,\" \"amazing,\" and \"so happy,\" indicating a clear positive sentiment."
}
{
  "sentiment": "negative",
  "confidence": 0.97,
  "reasoning": "The reviewer expresses clear disappointment and states the product is not worth the money, indicating a negative sentiment."
}
{
  "sentiment": "negative",
  "confidence": 0.86,
  "reasoning": "The reviewer describes the product as merely 'okay' and explicitly states they don't like it, indicating an overall unfavorable opinion."
}


In [72]:
class SentimentResponse(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float
    reasoning: str

sentiment_response1 = SentimentResponse.model_validate_json(review2)
print(sentiment_response1)

sentiment='negative' confidence=0.97 reasoning='The reviewer expresses clear disappointment and states the product is not worth the money, indicating a negative sentiment.'


In [ ]:
def validate_sentiment_reply(raw_reply: str) -> SentimentResponse | str:
    try:
        return SentimentResponse.model_validate_json(raw_reply)
    except ValidationError as e:
        raise ValueError(f"Invalid sentiment response: {e}")

In [76]:
def route_sentiment_reply(sentiment_response: SentimentResponse) -> str:
    if(sentiment_response.sentiment == "positive"):
        return "The review is positive"
    elif(sentiment_response.sentiment == "negative"):
        return "The review is negative, Please connect with the customer support team calling customer care number 1800-123-4567"
    else:
        return "The review is neutral"


In [77]:
sentiment_response1 = SentimentResponse.model_validate_json(review2)
print(route_sentiment_reply(sentiment_response1))

The review is negative, Please connect with the customer support team calling customer care number 1800-123-4567
